<a href="https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I'm picking this one because I don't yet know which of the data's signals actually carry real,
defensible information about visibility, clicks, engagement, or movement — and I'd rather find that
out before I build a ranked queue (Lane 2) or a CTR-gap score (Lane 4) on top of possibly-weak
signals. Lane 1's method, grouped effect sizes and comparisons, not a single correlation number, is also exactly what this data seems to need: Section 3 below shows a relationship (position vs.
CTR) whose raw linear correlation looks like noise but is actually large and real once you group by
tier. That is precisely the "reading big/small meaning into a single number" trap the lane guide
warns about, and precisely why I want to spend Week 1-2 doing signal analysis properly.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/vahagngrigoryan2006/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"

if IN_COLAB:
    import subprocess
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    # find the repo root from wherever this kernel started (works from notebooks/ or work/notebooks/)
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Working dir:", os.getcwd())
print(f"Starter dataset: {df.shape[0]:,} rows x {df.shape[1]} columns, "
      f"{df['client_id'].nunique()} pseudonymized clients")


Working dir: /content/flyrank-internship-ml/flyrank-internship-ml
Starter dataset: 30,000 rows x 44 columns, 32 pseudonymized clients


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Research question:** Which safe, observable content and search signals are reliably associated with a page's visibility (impressions/position), clicks (CTR), engagement, or movement (trend) and how large is each association, really?

**Decision this improves:** not one page's fate a decision one level up. Which signals the content
team should actually build its playbook and future prioritization rules around, versus which
"obvious" signals turn out to be weak, redundant, or already captured by something else. This is
the evidence base that a later ranked score (Lane 2/4-style) would stand on.

**Who acts, and what they do:** a content strategy lead reads the signal report and either (a) writes a new rule or
threshold into the editorial playbook, or (b) drops a heuristic the team currently uses that this
data shows doesn't actually hold up.

**Cost of a wrong call:**
- *A false signal accepted* - I report an association as real when it's actually noise or a
  confound: the team encodes a rule that doesn't work into guidelines applied across the whole
  content program, not just one page: every future refresh decision that rule touches inherits the
  mistake.
- *A real signal dismissed* - I under-read something because I only checked one summary number.
  Section 3 shows exactly this risk: the raw linear correlation between position and CTR is -0.08,
  which reads as "nothing here," but the grouped numbers say otherwise. Stopping at the weak
  correlation would throw away a real, large effect.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [7]:
has_pos = df[df["avg_position"] > 0]  # avg_position == 0 means "no data", not rank zero (gotcha)

# 1) The naive check: one linear correlation number
corr_pos_ctr = has_pos[["avg_position", "ctr"]].corr().iloc[0, 1]
print(corr_pos_ctr)

# 2) The real check: grouped by position tier
ctr_by_tier = has_pos.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)

# 3) Decline rate by freshness tier (the two tiers with real sample size on both sides)
df["is_declining"] = df["trend_direction"] == "down"
decline_by_freshness = df.groupby("freshness_tier")["is_declining"].agg(["mean", "count"])

print(f"1) Naive linear correlation between avg_position and ctr: {corr_pos_ctr:.2f} "
      f"-- on its own this reads as 'basically no relationship.'")
print()
print("2) But grouped by position tier, mean CTR is far from flat:")
print(ctr_by_tier.round(2).to_string())
best, worst = ctr_by_tier.iloc[0], ctr_by_tier.iloc[-1]
print(f"   -> top_3 pages average roughly {best / worst:.0f}x the CTR of 'deep' pages. The "
      f"relationship is real and large -- it's non-linear/tiered, so (1)'s single number badly "
      f"understates it.")
print()
fresh = decline_by_freshness.loc["0-30"]
stale = decline_by_freshness.loc["91-180"]
gap_pts = (stale["mean"] - fresh["mean"]) * 100
print(f"3) {stale['mean']*100:.1f}% of pages last updated 91-180 days ago are currently trending "
      f"down (n={int(stale['count']):,}), vs {fresh['mean']*100:.1f}% of pages updated in the last 30 "
      f"days (n={int(fresh['count']):,}) -- a ~{gap_pts:.0f}-point gap, on solid sample sizes both sides.")

-0.08014691740396772
1) Naive linear correlation between avg_position and ctr: -0.08 -- on its own this reads as 'basically no relationship.'

2) But grouped by position tier, mean CTR is far from flat:
position_tier
top_3       2.76
page_1      0.65
striking    0.32
page_3_5    0.22
deep        0.15
   -> top_3 pages average roughly 18x the CTR of 'deep' pages. The relationship is real and large -- it's non-linear/tiered, so (1)'s single number badly understates it.

3) 61.1% of pages last updated 91-180 days ago are currently trending down (n=9,171), vs 51.1% of pages updated in the last 30 days (n=20,480) -- a ~10-point gap, on solid sample sizes both sides.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can say by the end of this project:** which safe, observed signals are associated with
visibility, clicks, engagement, or movement in this data, and roughly how large each association
is - an effect size, not just "yes there's a link." Reported as observational, directional findings
("pages in the bottom position tier show much lower CTR than page-1 pages"), never as "moving up in
position causes X% more clicks."

**What I will never say:** that any of this proves a Google ranking factor; that a correlation here
means I could change one variable and cause the other to move — I watched, I did not run an
experiment or that `ai_sessions_90d` / `ai_traffic_pct` measure AI citations
or AI search visibility, they only count sessions where someone already clicked through from an AI
tool.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.